Exercise:

Use the unstructured texts to structured outputs.

1. MLOps Engineer
get the following:
* education_name
* program_description (summary)
* career_options (list of jobs)
* course_list
* length 

2. Kontakta oss
get the following:
* department
* opening_hours
* phone_time
* phone

export 2 JSON files

# Solution by apollon

In [ ]:
from pydantic_ai import Agent, TextContent
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import List
from pathlib import Path

load_dotenv(override=True)

mlops_raw_path = Path("mlops_raw.txt")

class EducationAnalyzer(BaseModel):
    education_name: str = Field(description="title of the education name with no description")
    program_description: str = Field(description="Broad description of the education")
    career_options: List = Field(description="Career paths that could be found in the education info and also other related career options")
    course_list: List = Field(description="List of courses named in the description of the education without naming the credits for the course")
    length: float = Field(description="Total length of the whole education")

education_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt="You are an education analyzer that analyzes education descriptions and give info for possible students that might be intrested in studing the analyzed education")

result = await education_agent.run(
    ["Analyze the attached education description and give me detailed summary", 
     TextContent(content=mlops_raw_path.read_text())], 
     output_type=EducationAnalyzer)

result.output.model_dump()

{'education_name': 'MLOps Engineer',
 'program_description': 'Build, deploy, and optimize machine learning models in real-world scenarios. Focuses on technical skills for implementing AI solutions across industries, including healthcare and big data. Includes hands-on practice with tools like TensorFlow, PyTorch, and DevOps pipelines.',
 'career_options': ['MLOps Engineer',
  'AI System Architect',
  'Data Pipeline Developer',
  'Cloud AI Specialist',
  'ML Model Deployment Engineer'],
 'course_list': ['Machine Learning Models and Algorithms',
  'MLOps and Cloud Platforms',
  'Python Programming for MLOps',
  'System Architecture and Tech Stacks for MLOps',
  'Security and Privacy for ML',
  'Edge Computing',
  'Linux Administration',
  'Continuous Integration and Delivery',
  'Database Handling',
  'AI-Enhanced Knowledge Strategy',
  'Business Administration',
  'Monitoring and Error Handling of ML Models',
  'Exam Paper'],
 'length': 2.5,
 'credits': 500.0}

In [19]:
import pandas as pd
df = pd.DataFrame(result.output)
df

,0,1
0,education_name,MLOps Engineer
1,program_description,"Build, deploy, and optimize machine learning m..."
2,career_options,"[MLOps Engineer, AI System Architect, Data Pip..."
3,course_list,"[Machine Learning Models and Algorithms, MLOps..."
4,length,2.5
5,credits,500.0


# Lecturer Solution

In [25]:
with open("mlops_raw.txt", "r") as file:
    mlops_text = file.read()

mlops_text[:150]

'Vill du jobba i skärningspunkten mellan AI, data och teknik? Som MLOps Engineer lär du dig att bygga, driftsätta och optimera maskininlärningsmodeller'

In [26]:
class Education(BaseModel):
    education_name: str = Field(description="Make sure to find this information in the text")
    program_description: str = Field(description="A 3-5 sentences summary about the program")
    career_options: list[str]
    courses: list[str]
    length: float = Field(description="Total study length of the program")

# language

In [29]:
education_extractor_agent = Agent(
    "openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
    system_prompt="""
    You are an expert in the Swedish education focused on Yrkeshögskola (Higher Vacational Education).
    Based on a text about an education, you will extract out relevant information.
"""
)

result = await education_extractor_agent.run(mlops_text, output_type=Education)
result.output

Education(education_name='MLOps Engineer - AI och maskininlärning', program_description='Utbildningen förbereder studenter att bygga, driftsätta och optimera maskininlärningsmodeller i verkligheten med fokus på tekniska och praktiska färdigheter. Studenter arbetar med verktyg som TensorFlow, PyTorch och Kubernetes, samt lär sig CI/CD-pipelines, säkerhet och edge computing. Uitbildningen kombinerar teori och praktik genom LIA-perioder och förbereder för internationella IT-rollers. Programmet pågår i 2,5 år (500 YH-poäng) och bedrivs främst på svenska med engelskt kursmaterial.', career_options=['MLOps Engineer'], courses=['AI-förstärkt kunskapsstrategi och hybrida AI-team', 'Databashantering', 'Edge computing', 'Kontinuerlig integration och leverans', 'Linux administration', 'Maskininlärningsmodeller och algoritmer', 'Maskininlärningsramverk', 'MLOps och molnplattformar', 'Python-programmering för MLOps', 'Systemarkitektur och teknikstackar för MLOps', 'Säkerhet och integritet', 'Överva

In [30]:
result.output.courses

['AI-förstärkt kunskapsstrategi och hybrida AI-team',
 'Databashantering',
 'Edge computing',
 'Kontinuerlig integration och leverans',
 'Linux administration',
 'Maskininlärningsmodeller och algoritmer',
 'Maskininlärningsramverk',
 'MLOps och molnplattformar',
 'Python-programmering för MLOps',
 'Systemarkitektur och teknikstackar för MLOps',
 'Säkerhet och integritet',
 'Övervakning och felhantering av ML-modeller']

In [31]:
with open("education.json", "w") as file:
    file.write(result.output.model_dump_json(indent=2))